# Moosic 05: Post the playlists to Spotify

**Goal of this notebook:** upload the playlists from notebook 04 to a Spotify account, so we can listen to them.

**Before you start:**
1. Create an app on the [Spotify developer dashboard](https://developer.spotify.com/dashboard)
   (tick **Web API**, redirect URI `http://127.0.0.1:8081/callback`). Since February 2026, the app owner needs **Spotify Premium**.
2. Copy `.env.example` to a new file called `.env` and paste your Client ID and Client Secret there.
   `.env` is listed in `.gitignore`, so **your secret never ends up on GitHub**.

**Input:** `results/moosic_playlists.csv` (from notebook 04) and `data/3_spotify_5000_songs.csv` (for the Spotify song ids)

## 1. Load our playlists and add back the Spotify song ids

In [19]:
import pandas as pd

In [20]:
# One row per song: name, artist, playlist number, description...
playlists = pd.read_csv('../results/moosic_playlists.csv')
playlists

,name,artist,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,playlist,description
0,Se Eu Quiser Falar Com Deus,Gilberto Gil,0.6580,0.25900,11,-13.141,0,0.0705,0.694,0.000059,0.9750,0.3060,110.376,256213,4,31,"Live feel, acoustic, calm"
1,Saudade De Bahia,Antônio Carlos Jobim,0.7420,0.39900,2,-12.646,1,0.0346,0.217,0.000002,0.1070,0.6930,125.039,191867,4,29,"Happy, acoustic"
2,"Canta Canta, Minha Gente",Martinho Da Vila,0.8510,0.73000,2,-11.048,1,0.3470,0.453,0.000063,0.1240,0.9050,93.698,152267,4,17,"Spoken / rap, danceable, slow"
3,Mulher Eu Sei,Chico César,0.7050,0.05020,4,-18.115,1,0.0471,0.879,0.000041,0.3860,0.5240,106.802,186227,4,31,"Live feel, acoustic, calm"
4,Rosa Morena,Kurt Elling,0.6510,0.11900,6,-19.807,1,0.0380,0.916,0.000343,0.1040,0.4020,120.941,273680,4,7,"Acoustic, calm, slow"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5108,"1812 Festival Overture, Op. 49: 1812 Overture,...",Pyotr Ilyich Tchaikovsky,0.2020,0.06940,3,-23.390,1,0.0473,0.917,0.715000,0.0832,0.0655,87.906,995000,4,28,"Quiet, calm, acoustic"
5109,"Winter Fragments pour ensemble instrumental, s...",Tristan Murail,0.1880,0.10100,6,-21.873,1,0.0442,0.855,0.426000,0.0694,0.0355,83.134,855000,4,28,"Quiet, calm, acoustic"
5110,"Schoenberg: 5 Orchestral Pieces, Op. 16: No. 3...",Arnold Schoenberg,0.0596,0.00093,9,-42.959,1,0.0434,0.951,0.969000,0.0772,0.0344,71.573,238187,3,28,"Quiet, calm, acoustic"
5111,"Serenade For Strings In E, Op.22, B. 52: 1. Mo...",Antonín Dvořák,0.1330,0.02080,4,-29.443,1,0.0419,0.883,0.505000,0.1110,0.0591,67.109,314307,4,28,"Quiet, calm, acoustic"


In [21]:
# We dropped the Spotify 'id' column during cleaning, so we get it back from the original file
songs = pd.read_csv('../data/3_spotify_5000_songs.csv')
songs.columns = songs.columns.str.strip()     # column names have extra spaces

In [22]:
# Text values have extra spaces too; without this, the merge below would find no matches
songs['name'] = songs['name'].str.strip()
songs['artist'] = songs['artist'].str.strip()
songs['id'] = songs['id'].str.strip()

In [23]:
# Same cleaning as before: one copy of each song
songs = songs.drop_duplicates(['name', 'artist'])
songs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5114 entries, 0 to 5234
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        5114 non-null   int64  
 1   name              5114 non-null   object 
 2   artist            5114 non-null   object 
 3   danceability      5114 non-null   float64
 4   energy            5114 non-null   float64
 5   key               5114 non-null   int64  
 6   loudness          5114 non-null   float64
 7   mode              5114 non-null   int64  
 8   speechiness       5114 non-null   float64
 9   acousticness      5114 non-null   float64
 10  instrumentalness  5114 non-null   float64
 11  liveness          5114 non-null   float64
 12  valence           5114 non-null   float64
 13  tempo             5114 non-null   float64
 14  type              5114 non-null   object 
 15  duration_ms       5114 non-null   int64  
 16  time_signature    5114 non-null   int64  
 17  

In [24]:
# Match on name + artist to add the Spotify id to each song in our playlists
playlists = playlists.merge(songs[['name', 'artist', 'id']], on=['name', 'artist'], how='left')

# Should be 0: every song found its id
playlists['id'].isna().sum()

np.int64(0)

In [25]:
playlists.head()

,name,artist,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,playlist,description,id
0,Se Eu Quiser Falar Com Deus,Gilberto Gil,0.658,0.2590,11,-13.141,0,0.0705,0.694,0.000059,0.975,0.306,110.376,256213,4,31,"Live feel, acoustic, calm",1n7JnwviZ7zf0LR1tcGFq7
1,Saudade De Bahia,Antônio Carlos Jobim,0.742,0.3990,2,-12.646,1,0.0346,0.217,0.000002,0.107,0.693,125.039,191867,4,29,"Happy, acoustic",5QGM1U0eCYrQuwSJwTm5Zq
2,"Canta Canta, Minha Gente",Martinho Da Vila,0.851,0.7300,2,-11.048,1,0.3470,0.453,0.000063,0.124,0.905,93.698,152267,4,17,"Spoken / rap, danceable, slow",0NLIFSZxPzQhCwnkn5PJYs
3,Mulher Eu Sei,Chico César,0.705,0.0502,4,-18.115,1,0.0471,0.879,0.000041,0.386,0.524,106.802,186227,4,31,"Live feel, acoustic, calm",3mXqOdlLE1k67WsAxryPFs
4,Rosa Morena,Kurt Elling,0.651,0.1190,6,-19.807,1,0.0380,0.916,0.000343,0.104,0.402,120.941,273680,4,7,"Acoustic, calm, slow",7bSzjzjTkWT2CkIPPdp0eA


## 2. Connect to Spotify

In [26]:
# Run once if needed:
# !pip install spotipy python-dotenv

import os
import pickle
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv

In [27]:
# Read the secrets from the .env file (never write them directly in the notebook!)
load_dotenv()

scope = 'playlist-modify-public playlist-modify-private'     # permission to create and change playlists
redirectUri = os.getenv('SPOTIFY_REDIRECT_URI', 'http://127.0.0.1:8081/callback')   # must match the dashboard
client_id = os.getenv('SPOTIFY_CLIENT_ID')
client_secret = os.getenv('SPOTIFY_CLIENT_SECRET')

In [28]:
token = SpotifyOAuth(scope=scope,
                     client_id=client_id,
                     client_secret=client_secret,
                     redirect_uri=redirectUri,
                     open_browser=True)          # True on your laptop, False in Google Colab

sp = spotipy.Spotify(auth_manager=token,
                     requests_timeout=10,        # give up after 10 seconds instead of hanging
                     retries=0,                  # show errors straight away, don't retry silently
                     status_retries=0)

# Sign-in test: a browser window opens the first time, then this prints your Spotify username
print(sp.current_user()['display_name'])

Omveer


## 3. Create the playlists
Set `TEST = True` to post only playlist 28 first and check it in Spotify.

Note: Spotify removed the old endpoints used in the course notebook (`user_playlist_create` and `.../tracks`) in
February 2026, so we call the new ones directly: `me/playlists` and `playlists/{id}/items`.

In [29]:
TEST = False
numbers = [28] if TEST else sorted(playlists['playlist'].unique())
numbers

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(23),
 np.int64(24),
 np.int64(25),
 np.int64(26),
 np.int64(27),
 np.int64(28),
 np.int64(29),
 np.int64(30),
 np.int64(31),
 np.int64(32),
 np.int64(33)]

In [30]:
playlist_collection = {}   # playlist name -> Spotify playlist id

for n in numbers:
    group = playlists[playlists['playlist'] == n]
    description = group['description'].iloc[0]
    name = f'Moosic {n:02d} · {description}'

    # Create an empty playlist for the signed-in user
    new_id = sp._post('me/playlists', payload={
        'name': name,
        'public': True,
        'description': f'{len(group)} songs, made by the Moosic prototype'})['id']

    # Add the songs. Spotify accepts max 100 songs per request, so we send them in batches of 100
    uris = [f'spotify:track:{i}' for i in group['id']]
    for start in range(0, len(uris), 100):
        sp._post(f'playlists/{new_id}/items', payload={'uris': uris[start:start + 100]})

    playlist_collection[name] = new_id
    print(f'{name}: {len(uris)} songs')

Moosic 00 · Quiet, calm, acoustic: 131 songs
Moosic 01 · Live feel, happy, danceable: 217 songs
Moosic 02 · Live feel, energetic, not danceable: 148 songs
Moosic 03 · Fast, not danceable, instrumental: 126 songs
Moosic 04 · Spoken / rap, danceable, happy: 194 songs
Moosic 05 · Instrumental, not danceable, electronic: 176 songs
Moosic 06 · All-round mix: 161 songs
Moosic 07 · Acoustic, calm, slow: 180 songs
Moosic 08 · Instrumental: 100 songs
Moosic 09 · Happy, danceable: 192 songs
Moosic 10 · Acoustic, calm, instrumental: 165 songs
Moosic 11 · Instrumental, not danceable, dark: 183 songs
Moosic 12 · Quiet, calm, acoustic: 86 songs
Moosic 13 · Fast, spoken / rap, happy: 132 songs
Moosic 14 · Danceable, happy: 201 songs
Moosic 15 · Calm, acoustic, slow: 97 songs
Moosic 16 · Spoken / rap, live feel, danceable: 159 songs
Moosic 17 · Spoken / rap, danceable, slow: 144 songs
Moosic 18 · All-round mix: 153 songs
Moosic 19 · Not danceable, fast, instrumental: 163 songs
Moosic 20 · Live feel: 1

## 4. Save the playlist ids and links

In [ ]:
# Keep the ids, so we can update the same playlists later instead of creating new ones
with open('../results/moosic_playlist_ids.pkl', 'wb') as f:
    pickle.dump(playlist_collection, f)

print('Saved', len(playlist_collection), 'playlists')

In [ ]:
# A table with a clickable link to every playlist
links = pd.DataFrame({
    'playlist': list(playlist_collection.keys()),
    'id': list(playlist_collection.values())
})
links['link'] = 'https://open.spotify.com/playlist/' + links['id']
links.to_csv('../results/moosic_playlist_links.csv', index=False)
links.head()